In [ ]:
%matplotlib widget

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from qualang_tools.units import unit
u = unit(coerce_to_integer=True)
from qm.qua import *
from qualang_tools.loops import from_array
import QM

# Exercise 0: Rabi oscillations period
Modify the pulse amplitude using the `amp` directive and observe how the Rabi oscillation frequency changes.

In [ ]:
# Parameters Definition
n_avg = 500  # The number of averages
# Pulse duration sweep (in clock cycles = 4ns) - must be larger than 4 clock cycles
t_min = 4 // 4
t_max = 1000 // 4
dt = 8 // 4
durations = np.arange(t_min, t_max, dt)
thermalization_time = 80 #in µs

###################
# The QUA program #
###################
with program() as qmprog:
    n = declare(int)  # QUA variable for the averaging loop
    t = declare(int)  # QUA variable for the qubit pulse duration
    I = declare(fixed)  # QUA variable for the measured 'I' quadrature
    Q = declare(fixed)  # QUA variable for the measured 'Q' quadrature
    I_st = declare_stream()  # Stream for the 'I' quadrature
    Q_st = declare_stream()  # Stream for the 'Q' quadrature

    with for_(n, 0, n < n_avg, n + 1):  # QUA for_ loop for averaging
        with for_(*from_array(t, durations)):  # QUA for_ loop for sweeping the pulse duration
            play("trigger","trigger")
            # Play the qubit pulse with a variable duration (in clock cycles = 4ns)
            play("pi"*amp(1.0), "qubit", duration=t)
            # Align the two elements to measure after playing the qubit pulse.
            align("qubit", "resonator")
            # Measure the state of the resonator
            measure(
                "readout",
                "resonator",
                dual_demod.full("cos", "sin", I),
                dual_demod.full("minus_sin", "cos", Q),
            )
            # Wait for the qubit to decay to the ground state
            wait(thermalization_time * u.us, "resonator")
            # Save the 'I' & 'Q' quadratures to their respective streams
            save(I, I_st)
            save(Q, Q_st)

    with stream_processing():
        # Cast the data into a 1D vector, average the 1D vectors together and store the results on the OPX processor
        I_st.buffer(len(durations)).average().save("I")
        Q_st.buffer(len(durations)).average().save("Q")

# Send the QUA program to the OPX, which compiles and executes it
job = QM.Job(qmprog)

# Non interacting version
# job = QM.JobSimple(qmprog)
# Wait for the job to finish
# job.wait()

In [ ]:
# Create plot
I,Q=job.get_results("I", "Q")
fig,ax = plt.subplots()
ax.plot(4*durations, I*1e4, 4*durations, Q*1e4)
ax.set_ylabel("Quadrature")
ax.set_xlabel("Pulse Duration [ns]")
ax.legend(('I','Q'))

# Exercise 1: Adjusting the Amplitude of the Gaussian $\pi$ Pulse

The objective is to find the microwave (MW) amplitude that rotates the qubit by $\pi$ using the predefined Gaussian pulse duration.

- Play the `x180` pulse on the qubit and vary its amplitude inside a loop using the `amp` directive.
- Plot the I and Q components of the readout signal as a function of the amplitude.


In [ ]:
# Parameters Definition
n_avg = 500  # The number of averages
amplitudes = np.arange(0, 1.2, 0.01)
thermalization_time = 80 #in µs

###################
# The QUA program #
###################
with program() as qmprog:
    n = declare(int)  # QUA variable for the averaging loop
    amplitude = declare(fixed)  
    I = declare(fixed)  # QUA variable for the measured 'I' quadrature
    Q = declare(fixed)  # QUA variable for the measured 'Q' quadrature
    I_st = declare_stream()  # Stream for the 'I' quadrature
    Q_st = declare_stream()  # Stream for the 'Q' quadrature
    n_st = declare_stream()  # Stream for the averaging iteration 'n'

    with for_(n, 0, n < n_avg, n + 1):  # QUA for_ loop for averaging
        with for_(*from_array(amplitude, amplitudes)): 
            # Play the qubit pulse with a variable duration (in clock cycles = 4ns)
            play("x180"*amp(amplitude), "qubit") # <--- Modify to change the amplitude with the loop variable
            # Align the two elements to measure after playing the qubit pulse.
            align("qubit", "resonator")
            # Measure the state of the resonator
            measure(
                "readout",
                "resonator",
                dual_demod.full("cos", "sin", I),
                dual_demod.full("minus_sin", "cos", Q),
            )
            # Wait for the qubit to decay to the ground state
            wait(thermalization_time * u.us, "resonator")
            # Save the 'I' & 'Q' quadratures to their respective streams
            save(I, I_st)
            save(Q, Q_st)

    with stream_processing():
        # Cast the data into a 1D vector, average the 1D vectors together and store the results on the OPX processor
        I_st.buffer(len(amplitudes)).average().save("I")
        Q_st.buffer(len(amplitudes)).average().save("Q")

# Send the QUA program to the OPX, which compiles and executes it
job = QM.Job(qmprog)

# Non interacting version
# job = QM.JobSimple(qmprog)
# Wait for the job to finish
# job.wait()

In [ ]:
# Create plot
I,Q=job.get_results("I", "Q")
fig,ax = plt.subplots()
ax.plot(amplitudes, I*1e4, amplitudes, Q*1e4)
ax.set_ylabel("Quadrature")
ax.legend(('I','Q'))
ax.set_xlabel("Pulse Amplitude Modifier")

In [ ]:
# Parameters Definition
n_avg = 500  # The number of averages
amplitudes = np.arange(0, 1.2, 0.01)
thermalization_time = 80 #in µs

###################
# The QUA program #
###################
with program() as qmprog:
    n = declare(int)  # QUA variable for the averaging loop
    amplitude = declare(fixed)  
    I = declare(fixed)  # QUA variable for the measured 'I' quadrature
    Q = declare(fixed)  # QUA variable for the measured 'Q' quadrature
    I_st = declare_stream()  # Stream for the 'I' quadrature
    Q_st = declare_stream()  # Stream for the 'Q' quadrature
    n_st = declare_stream()  # Stream for the averaging iteration 'n'

    with for_(n, 0, n < n_avg, n + 1):  # QUA for_ loop for averaging
        with for_(*from_array(amplitude, amplitudes)): 
            # Play the qubit pulse with a variable duration (in clock cycles = 4ns)
            play("y180"*amp(amplitude), "qubit") # <--- Modify to change the amplitude with the loop variable
            # Align the two elements to measure after playing the qubit pulse.
            align("qubit", "resonator")
            # Measure the state of the resonator
            measure(
                "readout",
                "resonator",
                dual_demod.full("cos", "sin", I),
                dual_demod.full("minus_sin", "cos", Q),
            )
            # Wait for the qubit to decay to the ground state
            wait(thermalization_time * u.us, "resonator")
            # Save the 'I' & 'Q' quadratures to their respective streams
            save(I, I_st)
            save(Q, Q_st)

    with stream_processing():
        # Cast the data into a 1D vector, average the 1D vectors together and store the results on the OPX processor
        I_st.buffer(len(amplitudes)).average().save("I")
        Q_st.buffer(len(amplitudes)).average().save("Q")

# Send the QUA program to the OPX, which compiles and executes it
job = QM.Job(qmprog)

# Non interacting version
# job = QM.JobSimple(qmprog)
# Wait for the job to finish
# job.wait()

In [ ]:
# Create plot
I,Q=job.get_results("I", "Q")
fig,ax = plt.subplots()
ax.plot(amplitudes, I*1e4, amplitudes, Q*1e4)
ax.set_ylabel("Quadrature")
ax.legend(('I','Q'))
ax.set_xlabel("Pulse Amplitude Modifier")

# Exercise 2: Determine the Lifetime of the Qubit Excited State ($T_1$)

- Write a sequence where the qubit is prepared in the $|1\rangle$ state at the beginning using an `x180` pulse.
- Delay the measurement by a variable time, modified inside a loop.
- Plot the I and Q components of the readout signal as a function of the delay.
- Estimate the qubit’s $T_1$ time.


In [ ]:
# Parameters Definition
n_avg = 2000  # The number of averages
# Pulse duration sweep (in clock cycles = 4ns) - must be larger than 4 clock cycles
t_min = 4 // 4
t_max = 200000 // 4
dt = 10000 // 4
durations = np.arange(t_min, t_max, dt)
thermalization_time = 80 #in µs

###################
# The QUA program #
###################
with program() as qmprog:
    n = declare(int)  # QUA variable for the averaging loop
    t = declare(int)  # QUA variable for the qubit pulse duration
    I = declare(fixed)  # QUA variable for the measured 'I' quadrature
    Q = declare(fixed)  # QUA variable for the measured 'Q' quadrature
    I_st = declare_stream()  # Stream for the 'I' quadrature
    Q_st = declare_stream()  # Stream for the 'Q' quadrature

    with for_(n, 0, n < n_avg, n + 1):  # QUA for_ loop for averaging
        with for_(*from_array(t, durations)):  # QUA for_ loop for sweeping the waiting time
            # Write the sequence
            # ....
            # Measure the state of the resonator
            measure(
                "readout",
                "resonator",
                dual_demod.full("cos", "sin", I),
                dual_demod.full("minus_sin", "cos", Q),
            )
            # Wait for the qubit to decay to the ground state
            wait(thermalization_time * u.us, "resonator")
            # Save the 'I' & 'Q' quadratures to their respective streams
            save(I, I_st)
            save(Q, Q_st)

    with stream_processing():
        # Cast the data into a 1D vector, average the 1D vectors together and store the results on the OPX processor
        I_st.buffer(len(durations)).average().save("I")
        Q_st.buffer(len(durations)).average().save("Q")

# Send the QUA program to the OPX, which compiles and executes it
job = QM.Job(qmprog)

# Non interacting version
# job = QM.JobSimple(qmprog)
# Wait for the job to finish
# job.wait()

In [ ]:
# Create plot
I,Q=job.get_results("I", "Q")
fig,ax = plt.subplots()
ax.plot(4*durations*1e-3, I*1e4, 4*durations*1e-3, Q*1e4)
ax.set_ylabel("Quadrature")
ax.set_xlabel("Pulse Delay [µs]")
ax.legend(('I','Q'))

# Exercise 3: Determine the Qubit Phase Coherence Time ($T_2$)

- Read the documentation on the [Ramsey sequence](https://qiskit-community.github.io/qiskit-experiments/manuals/characterization/t2ramsey.html).
- Write the Ramsey sequence in QUA using two `x90` pulses separated by a variable delay, modified inside a loop.
- Also include a `frame_rotation_2pi` command that rotates the reference frame by a phase proportional to the delay (use an addition or a multiplication; reading the documentation on [casting](https://docs.quantum-machines.co/latest/docs/API_references/qua/cast/) may be helpful).
- Plot the I and Q components of the readout signal as a function of the delay.
- Estimate the qubit’s $T_2$ time.


In [ ]:
# Parameters Definition
n_avg = 2000  # The number of averages
# Pulse duration sweep (in clock cycles = 4ns) - must be larger than 4 clock cycles
t_min = 4 // 4
t_max = 200000 // 4
dt = 10000 // 4
durations = np.arange(t_min, t_max, dt)
thermalization_time = 80 #in µs

###################
# The QUA program #
###################
with program() as qmprog:
    n = declare(int)  # QUA variable for the averaging loop
    t = declare(int)  # QUA variable for the qubit pulse duration
    I = declare(fixed)  # QUA variable for the measured 'I' quadrature
    Q = declare(fixed)  # QUA variable for the measured 'Q' quadrature
    I_st = declare_stream()  # Stream for the 'I' quadrature
    Q_st = declare_stream()  # Stream for the 'Q' quadrature

    with for_(n, 0, n < n_avg, n + 1):  # QUA for_ loop for averaging
        with for_(*from_array(t, durations)):  # QUA for_ loop for sweeping the waiting time
            # Write the sequence
            # ....
            # Measure the state of the resonator
            # Do not forget to wait for the end of the pulse on the qubit to perfom the measurement
            # (Use the align command)
            measure(
                "readout",
                "resonator",
                dual_demod.full("cos", "sin", I),
                dual_demod.full("minus_sin", "cos", Q),
            )
            # Wait for the qubit to decay to the ground state
            wait(thermalization_time * u.us, "resonator")
            # Save the 'I' & 'Q' quadratures to their respective streams
            save(I, I_st)
            save(Q, Q_st)

    with stream_processing():
        # Cast the data into a 1D vector, average the 1D vectors together and store the results on the OPX processor
        I_st.buffer(len(durations)).average().save("I")
        Q_st.buffer(len(durations)).average().save("Q")

# Send the QUA program to the OPX, which compiles and executes it
job = QM.Job(qmprog)

# Non interacting version
# job = QM.JobSimple(qmprog)
# Wait for the job to finish
# job.wait()

In [ ]:
# Create plot
I,Q=job.get_results("I", "Q")
fig,ax = plt.subplots()
ax.plot(4*durations*1e-3, I*1e4, 4*durations*1e-3, Q*1e4)
ax.set_ylabel("Quadrature")
ax.set_xlabel("Pulse Delay [µs]")
ax.legend(('I','Q'))

# Exercise 4: Spectroscopy of the $|1\rangle \rightarrow |2\rangle$ Transition

- Starting from the qubit spectroscopy sequence in `00_Spectroscopie`, write a sequence that performs spectroscopy of the $|2\rangle$ state.
